# Notebook 11: True 15% Test-Holdout R2 -- XGBoost vs. Spatial-CNN vs. Pixel-Transformer

**What this is:** the real chip-level R2 on each model's saved 15% test split
(`toy_model_split == "test"` in `split_manifest.jsonl` -- the exact split
frozen at training time, not re-derived), computed identically for all three
models so they're genuinely comparable. Unlike notebook 10 (5-tile spatial
inference R2), this reuses your colleague's *actual* trained artifacts:

- **XGBoost**: loads the saved `models/band_0X.json` boosters directly
  (`pace_vcf_toy.xgboost_baseline.XgboostModelSet`) and predicts on the
  saved test chips. This should exactly reproduce the `split=test` row
  already in that run's own `metrics_summary.csv` -- Cell 6 checks this
  and will warn loudly if it doesn't match, which would mean something
  about this notebook's preprocessing has drifted from theirs.
- **CNN / Pixel-Transformer**: no test-split R2 is logged anywhere on disk
  for these (Lightning's `trainer.test()` only records loss/mae/rmse,
  pooled across all bands, no R2 at all). This notebook reconstructs it by
  loading the saved `.ckpt` checkpoint via
  `pace_vcf_toy.xai.load_experiment()` (which rebuilds the exact trained
  network + reads the exact frozen test split) and running real inference.

**Important gotcha this avoids:** the repo actually has *two different R2
definitions* --
`xgboost_baseline.regression_statistics()`'s `coefficient_of_determination`
(conventional `1 - SS_res/SS_tot`) vs. its `correlation_r2` (squared Pearson
correlation, same quantity `validation_artifacts.ValidationArtifactWriter`'s
`r2` field reports elsewhere in the repo under a different name). Mixing the
two across models would silently produce non-comparable numbers -- this
notebook calls `regression_statistics()` once per model (getting both for
free) and uses **`correlation_r2`** (squared Pearson correlation) as the
headline metric throughout, per your choice. Note what that does and
doesn't capture: it measures how well predictions track truth *linearly*,
and is insensitive to a systematic bias/offset (a model that's consistently
too high by a constant amount can still score a high `correlation_r2`) --
`coefficient_of_determination` is still computed and saved alongside it in
every output file if you want to check for that separately.

**Scope:** all preprocessing (band exclusion, input normalization stats,
nodata/water masking) is read from each run's own `run_config.json` /
`normalization_stats.npz`, not hardcoded, so it matches what each model was
actually trained with. CPU-only by default (no GPU available in this
environment) -- inference on ~75 small (64x64) chips is fast enough on CPU;
set `DEVICE = "cuda"` in Cell 1 if running somewhere with one.


In [ ]:
# ## Cell 1: CONFIGURATION
# =============================================================================
# CONFIGURATION - EDIT THESE VALUES
# =============================================================================

from pathlib import Path

REPO_SRC = Path("/explore/nobackup/people/ajkerr1/PACE_Hyperspectral/pace_hyperspectral_vcf/toy_model/src")

COMPARISON_ROOT = Path(
    "/explore/nobackup/people/ajkerr1/PACE_Hyperspectral/model_outputs/"
    "4_model_comparison/fixed_tiling/date_2026_09_09-time_14_04_58-multi-model-comparison"
)

MODEL_SUBDIRS = {
    "xgboost": "XGBoost",
    "spatial-cnn": "Spatial-CNN",
    "pixel-transformer": "Pixel-Transformer",
}

DEVICE = "cpu"  # set to "cuda" if running on a GPU node

OUTPUT_DIR = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded")
print(f"  Comparison root: {COMPARISON_ROOT}")
print(f"  Device: {DEVICE}")


In [ ]:
#!pip install captum
#!pip install xgboost

In [ ]:
# ## Cell 2: Imports

import sys
sys.path.insert(0, str(REPO_SRC))

import json
import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

from pace_vcf_toy.records import JsonlManifestSource
from pace_vcf_toy.xgboost_baseline import (
    XgboostModelSet, XgboostDataConfig, ChipPixelExtractor, regression_statistics,
    raster_band_names,
)
from pace_vcf_toy.xai import load_experiment
from pace_vcf_toy.dataset import RasterChipDataset
from pace_vcf_toy.transforms import Compose, StandardizeInput

print("Imports complete (pace_vcf_toy loaded from colleague\'s repo)")


In [ ]:
# ## Cell 3: Locate Each Model\'s Run Directory

def find_model_run_dir(comparison_root: Path, model_subdir: str) -> Path:
    """Find the single nested date_XXX-time_XXX-<model>-<job_id> run dir."""
    model_dir = comparison_root / model_subdir
    if not model_dir.exists():
        raise FileNotFoundError(f"Model subdir not found: {model_dir}")
    candidates = sorted(
        d for d in model_dir.iterdir()
        if d.is_dir() and d.name.startswith("date_")
    )
    if not candidates:
        raise FileNotFoundError(f"No run directory found under: {model_dir}")
    if len(candidates) > 1:
        print(f"  NOTE: multiple run dirs under {model_dir.name}, using the last: {candidates[-1].name}")
    return candidates[-1]


run_dirs = {
    model_subdir: find_model_run_dir(COMPARISON_ROOT, model_subdir)
    for model_subdir in MODEL_SUBDIRS
}
for model_subdir, run_dir in run_dirs.items():
    print(f"{MODEL_SUBDIRS[model_subdir]}: {run_dir.name}")


In [ ]:
# ## Cell 4: XGBoost -- Real Inference on the Saved Test Split

def evaluate_xgboost_test_split(run_dir: Path, device: str = "cpu"):
    """Load the saved boosters and predict on the frozen test-split chips,
    matching the exact preprocessing (band exclusion, nodata/water masking)
    the run was trained/evaluated with. Returns (stats_df, targets, predictions)."""
    run_config = json.loads((run_dir / "run_config.json").read_text())

    model = XgboostModelSet.load(
        run_dir / "models",
        output_channels=run_config["output_channels"],
        feature_names=run_config["feature_names"],
        target_names=run_config["target_names"],
        device=device,
    )

    data_config = XgboostDataConfig(**run_config["data"])
    extractor = ChipPixelExtractor(data_config)

    records = JsonlManifestSource(run_dir / "split_manifest.jsonl").records()
    test_records = [r for r in records if (r.metadata or {}).get("toy_model_split") == "test"]
    print(f"  {len(test_records)} test chips")

    all_targets, all_predictions = [], []
    for record in test_records:
        batch = extractor.extract(record)
        predictions = model.predict(batch.features)
        all_targets.append(batch.targets)
        all_predictions.append(predictions)

    targets = np.concatenate(all_targets, axis=0)       # (n_pixels, output_channels)
    predictions = np.concatenate(all_predictions, axis=0)

    rows = []
    for band_index, band_name in enumerate(run_config["target_names"]):
        stats = regression_statistics(targets[:, band_index], predictions[:, band_index])
        rows.append({"target_band_name": band_name, **stats})
    return pd.DataFrame(rows), targets, predictions


xgboost_results, xgboost_targets, xgboost_predictions = evaluate_xgboost_test_split(run_dirs["xgboost"], device=DEVICE)
print(xgboost_results[["target_band_name", "valid_pixels", "correlation_r2", "rmse"]].to_string(index=False))


In [ ]:
# ## Cell 5: CNN / Pixel-Transformer -- Real Inference on the Saved Test Split

def evaluate_lightning_test_split(run_dir: Path, device: str = "cpu"):
    """Reconstruct the exact trained network from its checkpoint + run_config,
    load the frozen test-split chips with the same preprocessing (input
    normalization from this run's own normalization_stats.npz, nodata/water
    masking), and run real forward-pass inference. Returns (stats_df, targets, predictions)."""
    experiment = load_experiment(run_dir, checkpoint=None, device=device)
    module = experiment.module
    settings = experiment.data_settings

    test_records = experiment.records_by_split["test"]
    print(f"  {len(test_records)} test chips, checkpoint: {experiment.checkpoint_path.name}")

    stats_path = settings.normalization_stats_path or (run_dir / "normalization_stats.npz")
    standardize = StandardizeInput.from_npz(stats_path)

    dataset = RasterChipDataset(
        test_records,
        input_nodata_value=settings.input_nodata_value,
        label_nodata_value=settings.label_nodata_value,
        water_value=settings.water_value,
        exclude_water_from_loss=settings.exclude_water_from_loss,
        exclude_input_band_pattern=settings.exclude_input_band_pattern,
        transform=Compose((standardize,)),
    )

    output_channels = experiment.model_config.output_channels if hasattr(experiment.model_config, "output_channels") else None

    all_targets, all_predictions = [], []
    with torch.inference_mode():
        for i in range(len(dataset)):
            sample = dataset[i]
            image = sample["image"].unsqueeze(0).to(device)
            prediction = module(image)[0]  # (output_channels, H, W), already *target_scale

            target = sample["target"]              # (output_channels, H, W)
            mask = sample["target_mask"].bool()     # (1, H, W) or (output_channels, H, W)
            if mask.shape[0] == 1 and target.shape[0] > 1:
                mask = mask.expand_as(target)

            pred_np = prediction.cpu().numpy()
            target_np = target.numpy()
            mask_np = mask.numpy()

            n_bands = target_np.shape[0]
            flat_targets = target_np.reshape(n_bands, -1).T
            flat_predictions = pred_np.reshape(n_bands, -1).T
            flat_mask = mask_np.reshape(n_bands, -1).T[:, 0] if mask_np.shape[0] != n_bands else mask_np.reshape(n_bands, -1).T.any(axis=1)

            all_targets.append(flat_targets[flat_mask])
            all_predictions.append(flat_predictions[flat_mask])

    targets = np.concatenate(all_targets, axis=0)
    predictions = np.concatenate(all_predictions, axis=0)

    # target_names aren't persisted in the neural run_config (unlike XGBoost's),
    # but training derives them the same way: band descriptions on the label
    # raster of the first record (see experiment_runners.py / xgboost_baseline.raster_band_names).
    target_names = list(raster_band_names(test_records[0].label_path))
    if len(target_names) != targets.shape[1]:
        target_names = [f"band_{i+1}" for i in range(targets.shape[1])]

    rows = []
    for band_index in range(targets.shape[1]):
        band_name = target_names[band_index] if band_index < len(target_names) else f"band_{band_index+1}"
        stats = regression_statistics(targets[:, band_index], predictions[:, band_index])
        rows.append({"target_band_name": band_name, **stats})
    return pd.DataFrame(rows), targets, predictions


neural_results = {}
neural_targets = {}
neural_predictions = {}
for model_subdir in ("spatial-cnn", "pixel-transformer"):
    print(f"\n{MODEL_SUBDIRS[model_subdir]}:")
    df, targets, predictions = evaluate_lightning_test_split(run_dirs[model_subdir], device=DEVICE)
    neural_results[model_subdir] = df
    neural_targets[model_subdir] = targets
    neural_predictions[model_subdir] = predictions
    print(neural_results[model_subdir][["target_band_name", "valid_pixels", "correlation_r2", "rmse"]].to_string(index=False))


In [ ]:
# ## Cell 6: Sanity Check XGBoost Against the Run\'s Own Logged Numbers

logged = pd.read_csv(run_dirs["xgboost"] / "metrics_summary.csv")
logged_test = logged[logged["split"] == "test"][["target_band_name", "correlation_r2"]]
logged_test = logged_test.rename(columns={"correlation_r2": "logged_r2"})

check = xgboost_results[["target_band_name", "correlation_r2"]].rename(
    columns={"correlation_r2": "recomputed_r2"}
).merge(logged_test, on="target_band_name")
check["match"] = np.isclose(check["recomputed_r2"], check["logged_r2"], atol=1e-6)

print(check.to_string(index=False))
if check["match"].all():
    print("\nMATCH -- this notebook\'s preprocessing reproduces the run\'s own logged test R2 exactly.")
    print("The Spatial-CNN / Pixel-Transformer numbers above can be trusted with the same confidence.")
else:
    print("\nMISMATCH -- something in this notebook\'s preprocessing differs from the original run.")
    print("Do not trust the neural model numbers until this is resolved.")


In [ ]:
# ## Cell 7: Combined Summary Table + Save

all_results = []
for model_subdir, display_name in MODEL_SUBDIRS.items():
    df = xgboost_results if model_subdir == "xgboost" else neural_results[model_subdir]
    df = df.copy()
    df["model"] = display_name
    all_results.append(df)

combined = pd.concat(all_results, ignore_index=True)

print("=" * 70)
print("TRUE 15% TEST-HOLDOUT R2 (squared Pearson correlation)")
print("=" * 70)
summary = combined.pivot(index="target_band_name", columns="model", values="correlation_r2")
summary = summary[[m for m in MODEL_SUBDIRS.values() if m in summary.columns]]
print(summary.round(4).to_string())

combined_path = OUTPUT_DIR / "three_model_test_holdout_r2_full.csv"
combined.to_csv(combined_path, index=False)
print(f"\nSaved full results: {combined_path}")

summary_path = OUTPUT_DIR / "three_model_test_holdout_r2_summary.csv"
summary.to_csv(summary_path)
print(f"Saved summary table: {summary_path}")


In [ ]:
# ## Cell 8: Predicted vs. Actual Scatterplots (same style as 6h_pace_train_noprefix_legacy.ipynb)
%matplotlib inline
model_arrays = {
    "XGBoost": (xgboost_targets, xgboost_predictions),
    "Spatial-CNN": (neural_targets["spatial-cnn"], neural_predictions["spatial-cnn"]),
    "Pixel-Transformer": (neural_targets["pixel-transformer"], neural_predictions["pixel-transformer"]),
}

band_names = list(xgboost_results["target_band_name"])
n_bands = len(band_names)
n_models = len(model_arrays)

fig, axes = plt.subplots(n_bands, n_models, figsize=(5 * n_models, 5 * n_bands), squeeze=False)

for row, band_name in enumerate(band_names):
    for col, (model_name, (targets, predictions)) in enumerate(model_arrays.items()):
        y_true = targets[:, row]
        y_pred = predictions[:, row]
        r_pearson, _ = pearsonr(y_true, y_pred)
        r2 = r_pearson ** 2

        ax = axes[row, col]
        ax.scatter(y_true, y_pred, alpha=0.1, s=5)
        ax.plot([0, 100], [0, 100], 'r--', linewidth=2)
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.set_xlabel(f"Actual {band_name} (%)")
        ax.set_ylabel(f"Predicted {band_name} (%)")
        ax.set_title(f"{model_name} (R²={r2:.3f})")

plt.tight_layout()
fig_path = OUTPUT_DIR / "three_model_test_holdout_scatter.png"
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Saved: {fig_path}")
plt.show()